# Python Objects & Memory — Experimental Lab
**Software Development & Python · 90-minute seminar**

> What objects exist, who references them, and when do they disappear?

This lab reinforces **Inside Python: Objects, References & Runtime** and prepares
us for classes and the Python object model. No classes or third-party libraries today.

Work in pairs: **Predict → Run → Explain**. Draw objects as boxes and references as arrows.
Restart the kernel and run cells in order when you want a clean experiment.
Exercise outputs are intentionally blank in the saved notebook: predict before running.

| Section | Minutes |
| --- | ---: |
| 1. Names and objects: recap | 5 |
| 2. Function arguments and rebinding | 15 |
| 3. Containers store references | 7 |
| 4. Shallow vs deep copy | 15 |
| 5. Object memory | 10 |
| 6. Lifetime and reference counting | 10 |
| 7. Reference cycles | 7 |
| 8. Garbage collection | 6 |
| 9. Debugging challenge | 12 |
| 10. Summary | 3 |

The advanced challenge is optional within section 9. Instructor solutions are a
single removable section at the end.


In [25]:
import copy
import gc
import platform
import sys

print("Implementation:", platform.python_implementation())
print("Version:", platform.python_version())
print("Architecture:", platform.architecture()[0])


Implementation: CPython
Version: 3.12.0
Architecture: 64bit


## 1. Warm-up: Names and Objects
**5 minutes · recap**

We already saw these effects in **Python Survival Kit**.
Today we investigate the runtime model behind them.

### Predict first
How many list objects exist after each cell? Which names point to each one?


In [2]:
a = [1, 2]
b = a
print("Same object:", a is b)


Same object: True


In [3]:
b = [1, 2]
print("Same object:", a is b)
print("Equal contents:", a == b)


Same object: False
Equal contents: True


Assignment binds a name to an object. Rebinding `b` changes its arrow;
it does not change the old list.

```text
Before: a ──┐              After: a ──► list A [1, 2]
            ├─► [1, 2]            b ──► list B [1, 2]
        b ──┘
```

Our diagrams represent identity without relying on memory addresses.


## 2. Function Arguments and Rebinding
**15 minutes**

### Predict first
What happens to the caller's list? What object does the parameter `x` refer to?


In [4]:
def append_one(x):
    x.append(1)

a = []
append_one(a)
print(a)


[1]


### Predict first
Same caller, different operation: does assigning to `x` update `a`?


In [5]:
def replace_local(x):
    x = [1]
    print("Inside:", x)

a = []
replace_local(a)
print("Outside:", a)


Inside: [1]
Outside: []


Python passes object references into function parameters: a parameter is a new
local name, initially referring to the object supplied by the caller.
This is often called **call by sharing**.

```text
On entry:                 After x = [1]:
caller a ──┐              caller a ──► original list []
           ├─► list []    local  x ──► new list [1]
local  x ──┘
```

`append` changes a shared object. Assignment only rebinds the local name.
When the call ends, its ordinary local bindings go away; returned objects can survive.


### Predict first: is `+=` always mutation?
First use a list, then a tuple with a compatible operand.
Compare the object returned by the function with the caller's object.


In [6]:
def extend_list(x):
    x += [1]
    return x

a = []
result = extend_list(a)
print("Caller:", a, "Returned:", result)
print("Same object:", result is a)


Caller: [1] Returned: [1]
Same object: True


In [7]:
def extend_tuple(x):
    x += (1,)
    return x

a = (1, 2)
result = extend_tuple(a)
print("Caller:", a, "Returned:", result)
print("Same object:", result is a)


Caller: (1, 2) Returned: (1, 2, 1)
Same object: False


For a list, this `+=` extends the existing object. For a tuple, concatenation
produces a new tuple and rebinds the local name. The operator's behavior depends on type.

The operand matters too: `tuple += list` does not perform tuple concatenation.


In [8]:
a = (1, 2)
try:
    extend_list(a)  # This function tries tuple += list.
except TypeError:
    print("TypeError: a tuple cannot be concatenated with a list.")
print("Caller still:", a)


TypeError: a tuple cannot be concatenated with a list.
Caller still: (1, 2)


### Exercise 1 — Function Arguments
Before running, write the output and draw the arrows before and after `update`.
Identify the mutated object and the rebound name. Answers belong in your TODO cell.


In [ ]:
# TODO: predicted output = ...
# TODO: reference diagram = ...
# TODO: mutation vs rebinding = ...


In [ ]:
def update(x):
    x[0].append(10)
    x = [[100]]

a = [[1], [2]]
update(a)
print(a)


## 3. Containers Store References
**7 minutes**

A list holds references to Python objects. Its elements are not embedded copies
of arbitrary objects. Tuple elements and dictionary values also refer to objects.

```text
a ─► list
     ┌─────┬─────┬─────┐
     │ ref │ ref │ ref │
     └──│──┴──│──┴──│──┘
        ▼     ▼     ▼
      1000  2000  3000
```


In [11]:
a = [1000, 2000, 3000]
print(a)


[1000, 2000, 3000]


### Predict first
How many inner lists exist? Does an append through slot 0 affect slot 1?


In [12]:
inner = [1, 2]
a = [inner, inner]
print("Slots share an object:", a[0] is a[1])
a[0].append(3)
print(a)


Slots share an object: True
[[1, 2, 3], [1, 2, 3]]


```text
inner ────────────────► list [1, 2, 3]
                        ▲       ▲
a ─► outer list:       slot 0  slot 1
```

There are two list objects: the outer list and one inner list.
Two slots can point to the same object. Mutation follows an arrow to that object.


## 4. Shallow vs Deep Copy
**15 minutes**

### Predict first
After a shallow copy, are the outer lists identical? What about their first elements?


In [13]:
a = [[1, 2], [3, 4]]
b = a.copy()
print("Same outer list:", a is b)
print("Same first inner list:", a[0] is b[0])


Same outer list: False
Same first inner list: True


### Predict first
Which objects change when we assign through `b[0][0]`?


In [14]:
b[0][0] = 100
print("a:", a)
print("b:", b)


a: [[100, 2], [3, 4]]
b: [[100, 2], [3, 4]]


A shallow copy creates a new outer container with the same element references.

```text
a ─► outer A ─┬─► inner 1 [100, 2] ◄─┬─ outer B ◄─ b
              └─► inner 2 [3, 4]   ◄─┘
```

Changing an inner list is visible through both outer lists.


### Predict first
Will mutating a nested list in `c` affect `a`?


In [15]:
c = copy.deepcopy(a)
print("Same outer list:", c is a)
print("Same first inner list:", c[0] is a[0])
c[0].append(999)
print("a:", a)
print("c:", c)


Same outer list: False
Same first inner list: False
a: [[100, 2], [3, 4]]
c: [[100, 2, 999], [3, 4]]


For this nested-list graph, `deepcopy` creates new lists throughout the graph.
It recursively copies objects where appropriate; immutable objects may be reused.
It uses a record of already copied objects to handle cycles and internal sharing.
It does not copy raw bytes or promise complete independence for every possible object.

See the standard-library [copy documentation](https://docs.python.org/3/library/copy.html).


### Exercise 2 — Shallow Copy
1. Predict what `original` contains after the append.
2. Draw the paths from both outer lists to Alice's language list.
3. Create a backup whose nested lists are independent.

Put your fix in the TODO cell; solutions are at the end.


In [ ]:
original = [
    ["Alice", ["Python", "C++"]],
    ["Bob", ["Python"]],
]
backup = original.copy()
backup[0][1].append("Rust")
print(original)


In [ ]:
# TODO: predicted contents and explanation = ...
# TODO: create independent_backup from original.
# TODO: mutate its nested language list and check original is unchanged.


## 5. How Much Memory Does a Python Object Use?
**10 minutes**

### Predict first
Is an integer object just its numeric payload? Does a larger value take more space?


In [24]:
print("Small integer:", sys.getsizeof(1), "bytes")
print("Large integer:", sys.getsizeof(10**100), "bytes")


Small integer: 28 bytes
Large integer: 72 bytes


In CPython, an integer object includes type information, reference-management
metadata, and its numeric payload. Unlike a fixed-width C `int`, Python integers
support arbitrarily large values subject to available memory.

Exact sizes depend on implementation, version, architecture, and build details.
These observations describe **this runtime**, not universal byte counts.


### Predict first
What does `getsizeof` measure when the object is a list?


In [19]:
print("Empty list:", sys.getsizeof([]), "bytes")
print("One slot:", sys.getsizeof([1]), "bytes")
print("Three slots:", sys.getsizeof([1, 2, 3]), "bytes")


Empty list: 56 bytes
One slot: 64 bytes
Three slots: 88 bytes


In [20]:
x = [0] * 1000
print("List itself:", sys.getsizeof(x), "bytes")
print("One referenced integer:", sys.getsizeof(x[0]), "bytes")
print("All slots refer to the same object:", all(item is x[0] for item in x))


List itself: 8056 bytes
One referenced integer: 28 bytes
All slots refer to the same object: True


`sys.getsizeof(x)` reports the object's **shallow footprint**: the list's storage,
including reference slots, generally excludes the objects those slots point to.
It is not the total size of the reachable graph.

```text
1000 list slots ───► one integer object 0
```

A graph estimate would add the sizes of distinct reachable objects to the list's
size. Adding one integer size per slot counts this shared integer 1000 times.
That estimate still differs from process memory and allocator usage.

See [sys.getsizeof](https://docs.python.org/3/library/sys.html#sys.getsizeof).


### Predict first: same slot count, different reachable objects
Can a container hide a large payload from its shallow size?


In [21]:
small_payload = [b"x"]
large_payload = [b"x" * 100_000]
print("Outer list bytes:", sys.getsizeof(small_payload), sys.getsizeof(large_payload))
print("Payload bytes:", sys.getsizeof(small_payload[0]), sys.getsizeof(large_payload[0]))


Outer list bytes: 64 64
Payload bytes: 34 100033


Both outer lists contain one reference slot. The payload sizes differ greatly.
Always specify whether you are measuring one object or an object graph.


## 6. Object Lifetime and Reference Counting
**10 minutes · CPython implementation details**

Deleting a name removes a binding, not necessarily its object.
CPython uses reference counting as a central lifetime mechanism. In ordinary cases,
a zero count permits prompt reclamation; exact timing is not a Python language guarantee.

### Predict first
How should adding and deleting the alias change the observed count?


In [ ]:
a = []
if platform.python_implementation() == "CPython":
    before = sys.getrefcount(a)
    b = a
    with_alias = sys.getrefcount(a)
    del b
    after = sys.getrefcount(a)
    print("Before / with alias / after:", before, with_alias, after)
else:
    print("Reference-count experiment is intended for CPython.")


Before / with alias / after: 2 3 2


`sys.getrefcount(a)` temporarily adds a reference while receiving its argument.
The result is typically one higher than a diagram of ordinary bindings suggests.
Focus on **changes**, not a promised absolute number. Notebook history, debuggers,
and runtime internals can retain references; we print numbers rather than display the list itself.

Some CPython objects are immortal and their reported counts are not ordinary reference tallies.
We use a fresh list, not cached integers, for this experiment.

See [sys.getrefcount](https://docs.python.org/3/library/sys.html#sys.getrefcount).


### Exercise 3 — Reference Graph
Draw all visible references to the empty list below.
How many are there before `del a`? How many remain afterward?
Does removing the name make the object unreachable? Explain the paths that remain.


In [ ]:
a = []
b = [a]
c = {"value": a}


In [ ]:
# TODO: visible incoming references before del = ...
# TODO: draw the object graph before and after deleting a.
# TODO: will the list disappear? Why?


In [ ]:
del a
print("Same reachable list:", b[0] is c["value"])
print("Reachable contents:", b[0])


## 7. Reference Cycles
**7 minutes**

### Predict first
What graph does this code create? After external bindings disappear,
can reference counting alone reduce the two counts to zero?


In [31]:
a = []
b = []
a.append(b)
b.append(a)
print("A's first element is B:", a[0] is b)
print("B's first element is A:", b[0] is a)


A's first element is B: True
B's first element is A: True


```text
       a                         b
       │                         │
       ▼                         ▼
     list A ─────────────────► list B
            ◄─────────────────
```

Each list has an incoming reference from the other, in addition to its name.


In [32]:
del a
del b
print("Both external bindings have been removed.")


Both external bindings have been removed.


At the graph level, removing those bindings leaves mutual references but no path
from our live names to either list. Pure reference counting cannot reclaim that cycle:
neither incoming count becomes zero merely by deleting the names.

CPython's cyclic collector can reclaim unreachable cycles. It might already have
run by the time we discuss this cell; we make no claim about the exact time of reclamation.
Keeping an extra alias to inspect the lists would keep them reachable!


## 8. Garbage Collection
**6 minutes · CPython implementation details**

CPython combines reference counting with collection of unreachable cycles.
Having a cycle is not enough to make something garbage: a reachable cycle is still live.

### Predict first
Can we assume exactly two objects will be collected now?


In [33]:
collected = gc.collect()
print("Objects reported collected by this call:", collected)


Objects reported collected by this call: 2


The result concerns the runtime as a whole. It may include unrelated garbage;
our cycle may already have been collected. Do not assert a fixed result.

**Normal Python programs should usually let the runtime manage this.**
We call `gc.collect()` here to explore the mechanism, not as a routine programming requirement.
Reclaiming objects also does not guarantee an immediate drop in operating-system memory usage.

See the [gc documentation](https://docs.python.org/3/library/gc.html).


### Optional observation: tracked is not the same as garbage
A tracked object participates in cyclic-GC bookkeeping. It can still be reachable.
Predict which of the following can hold references that form cycles.


In [10]:
print("Integer tracked:", gc.is_tracked(42))
print("Fresh list tracked:", gc.is_tracked([]))
print("Dictionary tracked:", gc.is_tracked({"value": []}))


Integer tracked: False
Fresh list tracked: True
Dictionary tracked: True


Tracking decisions depend on implementation and object contents. These results do
not tell us whether an object is unreachable, nor when it will disappear.


## 9. Debugging Challenge
**12 minutes**

### Exercise 4 — An Unexpectedly Changed Input
`prepare` should return an independent nested copy, leaving its input unchanged.

1. Predict both printed values before running.
2. Explain the bug using a reference graph, including the function parameter.
3. Implement `prepare_fixed` in the TODO cell and check nested independence.
4. Predict the effect of replacing `processed[0]` in the second experiment.

Distinguish changing a shared nested object from replacing a slot in one outer list.


In [ ]:
def prepare(data):
    result = data.copy()
    result[0].append("processed")
    return result

original = [["item1"], ["item2"]]
processed = prepare(original)
print("Original:", original)
print("Processed:", processed)


In [ ]:
# TODO: predicted outputs = ...
# TODO: object graph and cause of the bug = ...
# TODO: define prepare_fixed(data).
# TODO: use fresh input and check that both nested lists are independent.


### Predict first: replace a slot
Does the following change `original[0]`? Which arrow is replaced?


In [ ]:
processed[0] = ["replaced"]
print("Original:", original)
print("Processed:", processed)


In [ ]:
# TODO: explain mutation of an inner object vs replacing an outer-list slot.


### Exercise 5 — Optional Advanced Challenge
For the graph below, predict these four expressions without running them:

```python
a[0] is a[1]
b[0] is a[0]
c[0] is a[0]
c[0] is c[1]
```

Then test your predictions. Does deep copying remove all internal sharing,
or preserve the structure of sharing inside a new graph?


In [ ]:
x = []
a = [x, x]
b = a.copy()
c = copy.deepcopy(a)
# TODO: four predictions and a diagram = ...
# TODO: print the four identity comparisons and explain each result.


## 10. Summary
**3 minutes**

- Variables are names bound to objects.
- Function parameters are new names initially referring to the caller's objects.
- Containers usually hold references to other Python objects.
- A shallow copy creates a new outer object while sharing nested objects.
- A deep copy recursively copies a graph where appropriate, preserving internal sharing.
- `getsizeof` measures the object itself, not its entire reachable graph.
- In CPython, reference counting is central to object lifetime.
- Unreachable cycles require an additional collection mechanism.

```text
    names (live roots)
      │
      ▼
    objects ◄──── containers
      ▲              │
      └──────────────┘

    reachable objects   → alive
    unreachable objects → garbage, eligible for reclamation
```

Reachability is a useful conceptual model. Exact management mechanisms and timing
depend on the Python implementation. Next time, classes will let us define our own
objects within this same model.

**Exit prompt:** Explain why `del name`, mutating an object, and replacing a container
slot are three different operations. Use arrows rather than a memorized output.
